# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dhruvaiyer2-netizen/FlyRankMLProject/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q duckdb huggingface_hub scikit-learn

from huggingface_hub import login
login()  # paste your HF token

import duckdb, numpy as np, pandas as pd
from huggingface_hub import HfFileSystem
duckdb.register_filesystem(HfFileSystem())

FACT = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
DIM  = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'

# Page-month aggregates (ratio-of-sums for ctr and position)
monthly = duckdb.sql(f"""
    SELECT month, content_hash_id, client_hash_id,
      SUM(gsc_impressions) AS impressions,
      SUM(gsc_clicks) AS clicks,
      SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
      100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM '{FACT}'
    WHERE month IN ('2026-03', '2026-04')
    GROUP BY month, content_hash_id, client_hash_id
""").df()

dim = duckdb.sql(f"""
    SELECT content_hash_id, word_count, main_intent, content_created_date, is_deleted
    FROM '{DIM}'
""").df()

MIN_IMP = 500
TIER_BINS = [0, 3, 10, 20, float("inf")]
TIER_LABELS = ["1-3", "4-10", "11-20", "21+"]

def prep(df):
    """Volume filter, position tier, tier-relative CTR gap (the Week 4 rule's score)."""
    df = df[df["impressions"] >= MIN_IMP].copy()
    df["tier"] = pd.cut(df["avg_position"], TIER_BINS, labels=TIER_LABELS)
    df["tier_median_ctr"] = df.groupby("tier", observed=True)["ctr"].transform("median")
    df["ctr_gap_score"] = df["tier_median_ctr"] - df["ctr"]
    return df

mar = prep(monthly[monthly["month"] == "2026-03"])
apr = prep(monthly[monthly["month"] == "2026-04"])

# Label comes from APRIL only: did the page underperform its tier the following month?
apr_label = apr[["content_hash_id", "client_hash_id", "ctr_gap_score"]].rename(
    columns={"ctr_gap_score": "apr_gap"})
data = mar.merge(apr_label, on=["content_hash_id", "client_hash_id"], how="inner")
data["y"] = (data["apr_gap"] > 0).astype(int)

# Content metadata; drop deleted pages
data = data.merge(dim, on="content_hash_id", how="left")
data = data[data["is_deleted"] != True].copy()

# Features: everything below is knowable by 2026-03-31
data["word_count_missing"] = data["word_count"].isna().astype(int)
data["word_count_filled"] = data["word_count"].fillna(-1)
data["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(data["content_created_date"])).dt.days.clip(lower=0)
data = pd.concat([data, pd.get_dummies(data["main_intent"], prefix="intent", dtype=int)], axis=1)
intent_cols = [c for c in data.columns if c.startswith("intent_")]

FEATURES = ["ctr", "avg_position", "impressions", "ctr_gap_score",
            "word_count_filled", "word_count_missing", "content_age_days"] + intent_cols

print(f"Pages with >=500 impressions in BOTH March and April: {len(data)}")
print(f"Share underperforming in April (base rate): {data['y'].mean():.3f}")
print(f"Features: {FEATURES}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with >=500 impressions in BOTH March and April: 51494
Share underperforming in April (base rate): 0.513
Features: ['ctr', 'avg_position', 'impressions', 'ctr_gap_score', 'word_count_filled', 'word_count_missing', 'content_age_days', 'intent_commercial', 'intent_informational', 'intent_navigational', 'intent_transactional']


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm comparing three things on the same split: the Week 4 rule (baseline), Logistic Regression (simple, interpretable), and Random Forest (captures non-linear interactions). The task is predicting whether a page that is visible in March will still underperform its position tier in April, using only March information. Random Forest is my main candidate because the CTR-vs-position relationship is non-linear and interacts with volume. Logistic Regression is included so I can tell whether the forest's extra complexity actually earns its keep. I'm skipping Gradient Boosting: it adds tuning risk without a clear reason to expect a gain over the forest on this feature set. The success bar is beating the Week 4 rule on the same test pages, not winning on complexity.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The split is grouped by client and time-aware by construction. Features are March, the label is April, so nothing from the label's period feeds the inputs. Grouping by client means the model is tested on clients it never trained on, which is the realistic deployment question. A random split would let client-specific quirks leak across train and test. One caveat to state: tier medians are computed over each whole month, including test pages. It's a summary statistic, not a label, so I treat it as minor.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(data, groups=data["client_hash_id"]))
train, test = data.iloc[tr_idx].copy(), data.iloc[te_idx].copy()

print(f"Train: {len(train)} rows, {train['client_hash_id'].nunique()} clients, base rate {train['y'].mean():.3f}")
print(f"Test:  {len(test)} rows, {test['client_hash_id'].nunique()} clients, base rate {test['y'].mean():.3f}")
print(f"Client overlap (must be 0): {len(set(train['client_hash_id']) & set(test['client_hash_id']))}")

Train: 25319 rows, 23 clients, base rate 0.505
Test:  26175 rows, 11 clients, base rate 0.521
Client overlap (must be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

On a client-grouped test set (base rate 0.521), the Week 4 rule is a strong baseline (ROC-AUC 0.822), since a page that underperforms its tier in March mostly still does in April. Random Forest improves on it modestly but consistently: AUC 0.837 vs 0.822 on the main split, and it wins in all five client-grouped folds, with a mean gain of about +0.014. At the top of the queue it reaches precision@500 of 0.982, versus 0.880 for the rule once its tied scores are broken by impressions (0.820 with arbitrary ties). Logistic Regression does not beat the rule on AUC (0.818). The most defensible claim is that the forest gives a small, reliable ranking improvement, with its clearest gain where reviewer attention is spent, at the top of the list.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X_train, y_train = train[FEATURES], train["y"].values
X_test,  y_test  = test[FEATURES],  test["y"].values

# Baseline = Week 4 rule applied to March: flag if March CTR is below its tier median
base_score = test["ctr_gap_score"].values
base_pred = (base_score > 0).astype(int)

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logit.fit(X_train, y_train)

rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

def evaluate(name, pred, score, k=500):
    top_k = np.argsort(-score)[:k]
    return {"method": name,
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, score),
            f"precision@{k}": y_test[top_k].mean()}

rows = [
    evaluate("week4_rule (baseline)", base_pred, base_score),
    evaluate("logistic_regression", logit.predict(X_test), logit.predict_proba(X_test)[:, 1]),
    evaluate("random_forest", rf.predict(X_test), rf.predict_proba(X_test)[:, 1]),
]
order = np.lexsort((-test["impressions"].values, -base_score))   # primary: score, secondary: impressions
tb_score = np.empty(len(test)); tb_score[order] = np.arange(len(test), 0, -1)

results = pd.DataFrame([
    evaluate("week4_rule (baseline)", base_pred, base_score),
    evaluate("week4_rule + impressions tiebreak", base_pred, tb_score),
    evaluate("logistic_regression", logit.predict(X_test), logit.predict_proba(X_test)[:, 1]),
    evaluate("random_forest", rf.predict(X_test), rf.predict_proba(X_test)[:, 1]),
]).set_index("method").round(3)
print(f"Test base rate: {y_test.mean():.3f}")
results

Test base rate: 0.521


,precision,recall,f1,roc_auc,precision@500
method,,,,,
week4_rule (baseline),0.780,0.718,0.748,0.822,0.820
week4_rule + impressions tiebreak,0.780,0.718,0.748,0.822,0.880
logistic_regression,0.723,0.848,0.781,0.818,0.876
random_forest,0.747,0.823,0.783,0.837,0.982


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The forest leans almost entirely on March's own CTR gap (ctr_gap_score, AUC drop 0.157 when shuffled; ctr second at 0.047, nearly a duplicate). Position and impressions add a little, while word count, content age and intent add essentially nothing, so the model is mostly re-learning the rule and the small gain reflects that. It errs on 23.7% of test pages, and about 61% of those errors are false positives, meaning pages flagged as underperforming that recovered in April. Error is nearly flat across position tiers (22.5-24.6%) but falls with volume, from 28.3% at 500-1k impressions to 18.2% above 5k, which matches the Week 4 finding that low-volume CTR is noisy. No single client drives the errors (22-29% across the largest clients).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix

# What does the model lean on?
perm = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, scoring="roc_auc", n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
print("Permutation importance (drop in AUC when shuffled):")
print(imp.round(4).head(8))

# Where is it wrong?
test["rf_pred"] = rf.predict(X_test)
test["error"] = (test["rf_pred"] != test["y"]).astype(int)
print("\nConfusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(test["y"], test["rf_pred"]))

print("\nError rate by position tier:")
print(test.groupby("tier", observed=True)["error"].agg(error_rate="mean", n="size").round(3))

test["vol_bucket"] = pd.cut(test["impressions"], [499, 1000, 5000, np.inf], labels=["500-1k", "1k-5k", "5k+"])
print("\nError rate by March volume:")
print(test.groupby("vol_bucket", observed=True)["error"].agg(error_rate="mean", n="size").round(3))

print("\nWorst 5 clients by error rate (min 100 pages):")
by_client = test.groupby("client_hash_id")["error"].agg(error_rate="mean", n="size")
print(by_client[by_client["n"] >= 100].sort_values("error_rate", ascending=False).head(5).round(3))

Permutation importance (drop in AUC when shuffled):
ctr_gap_score          0.1565
ctr                    0.0472
impressions            0.0174
avg_position           0.0097
content_age_days       0.0021
word_count_filled      0.0018
word_count_missing     0.0002
intent_navigational   -0.0000
dtype: float64

Confusion matrix [rows=true, cols=pred]:
[[ 8747  3793]
 [ 2416 11219]]

Error rate by position tier:
       error_rate      n
tier                    
1-3         0.225   3091
4-10        0.232  11806
11-20       0.245   4792
21+         0.246   6486

Error rate by March volume:
            error_rate      n
vol_bucket                   
500-1k           0.283   4164
1k-5k            0.252  14594
5k+              0.182   7417

Worst 5 clients by error rate (min 100 pages):
                         error_rate      n
client_hash_id                            
client_fef1a8f436438636       0.285   4011
client_73cda7b4e4f265ea       0.236  12489
client_3197e6291363b4db       0.233    15

Limitations:
The model covers only pages with at least 500 impressions in both months.
It uses one March-to-April transition, so another month could behave differently.
Test data is concentrated in a few large clients.
The gain over the rule is real but small, and a team could reasonably prefer the rule plus a tie-break for its simplicity.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.